In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Literal , Annotated
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field
import operator

load_dotenv()

generator_llm = ChatGroq(model="openai/gpt-oss-20b")
optimizer_llm = ChatGroq(model="openai/gpt-oss-20b")
evaluator_llm = ChatGroq(model="openai/gpt-oss-20b")


class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(
        ...,
        description="Final evaluation result."
    )
    feedback: str = Field(
        ...,
        description="Constructive feedback for the tweet."
    )


sutructured_evaluator_llm = evaluator_llm.with_structured_output(
    TweetEvaluation,
    method="json_schema"
)


class Tweet(TypedDict):
    topic: str
    tweet: str
    feedback: str
    iteration: int
    max_iteration: int
    evaluation: Literal["approved", "needs_improvement"]
    tweet_history = Annotated[list[str], operator.add]
    feedback_history = Annotated[list[str], operator.add]



def generate_tweet(state: Tweet):
    messages = [
        SystemMessage(
            content="You are a funny and clever Twitter/X influencer."
        ),

        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day English.
- This is version {state['iteration'] + 1}.
""")
    ]

    response = generator_llm.invoke(messages)

    return {
        "tweet": response.content , "tweet_history": [response]
    }


def evaluate_tweet(state: Tweet):
    messages = [
        SystemMessage(
            content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."
        ),

        HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?
2. Humor – Did it genuinely make you smile, laugh, or chuckle?
3. Punchiness – Is it short, sharp, and scroll-stopping?
4. Virality Potential – Would people retweet or share it?
5. Format – Is it a well-formed tweet, not a setup-punchline joke, not a Q&A joke, and under 280 characters?

Auto-reject if:
- It's written in question-answer format.
- It exceeds 280 characters.
- It reads like a traditional setup-punchline joke.
- It ends with generic, throwaway, or deflating lines that weaken the humor.

Respond using the required structured format.
""")
    ]

    response = sutructured_evaluator_llm.invoke(messages)

    return {
        "evaluation": response.evaluation,
        "feedback": response.feedback,
        "feedback_history":[response.feedback]
    }


def optimize_tweet(state: Tweet):
    messages = [
        SystemMessage(
            content="You punch up tweets for virality and humor based on given feedback."
        ),

        HumanMessage(content=f"""
Improve the tweet based on this feedback:

"{state['feedback']}"

Topic:
"{state['topic']}"

Original Tweet:
{state['tweet']}

Rewrite it as a short, viral-worthy tweet.

Rules:
- Avoid Q&A style.
- Stay under 280 characters.
- Make it funnier.
- Make it more original.
- Make the punchline stronger.
""")
    ]

    response = optimizer_llm.invoke(messages).content

    iteration = state["iteration"] + 1

    return {
        "tweet": response,
        "iteration": iteration,
        "tweet_history": [response]
    }


def route_evaluation(state: Tweet):
    if (
        state["evaluation"] == "approved"
        or state["iteration"] >= state["max_iteration"]
    ):
        return "approved"

    else:
        return "needs improvement"


graph = StateGraph(Tweet)

graph.add_node("generate", generate_tweet)
graph.add_node("evaluator", evaluate_tweet)
graph.add_node("optimize", optimize_tweet)

graph.add_edge(START, "generate")

graph.add_edge("generate", "evaluator")

graph.add_conditional_edges(
    "evaluator",
    route_evaluation,
    {
        "approved": END,
        "needs improvement": "optimize"
    }
)

graph.add_edge("optimize", "evaluator")


work_flow = graph.compile()


initial_state = {
    "topic": "hgjh",
    "iteration": 1,
    "max_iteration": 5
}


result = work_flow.invoke(initial_state)


print("FINAL TWEET:")
print(result["tweet"])

print("\nEVALUATION:")
print(result["evaluation"])

print("\nFEEDBACK:")
print(result["feedback"])

print("\nITERATION:")
print(result["iteration"])



FINAL TWEET:
Typed “hgjh”, autocorrect turned it into “happiness”. My keyboard just handed me a self‑help book. Guess my phone finally decided to fix my mood before I even try. 🤦‍♂️ #KeyboardTherapist

EVALUATION:
approved

FEEDBACK:
The tweet lands well—original enough to feel fresh, with a quick, witty observation about autocorrect that’s likely to get a few chuckles. It’s punchy and under the character limit, and the hashtag gives it a clear, shareable hook. Minor tweak: consider tightening the wording to shave a few characters for extra brevity, but overall it’s solid for retweets.

ITERATION:
2

FEEDBACK_HISTORY:


KeyError: 'feedback_history'